# Example 3: Physical Room from DataRES Dataset

This notebook shows how to create a **physical room** in PyRES by loading real measured data from the **DataRES** dataset.

DataRES (doi:10.5281/zenodo.15165524) is an accompanying dataset for PyRES. It contains impulse responses measured in various physical spaces that host a Reverberation Enhancement System (RES). For each room, the dataset provides:
- Positions of all transducers: stage emitters, system microphones, system loudspeakers, audience receivers.
- The full set of room impulse responses (RIRs) between every pair of transducer groups.

The `PhRoom_dataset` class handles the loading of these data and presents them in the same unified interface as the synthetic room classes (e.g., `PhRoom_wgn`).

Let’s begin by importing the necessary modules.

## 1. Imports and Path Setup

Add the parent directory to the Python path so that PyRES can be imported (assuming the notebook is in the `examples/` folder). We also import `mag2db` from `flamo.functional` to convert linear values to decibels.

In [ ]:
import sys
import os
# Add parent directory to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt
from flamo.functional import mag2db
from PyRES.physical_room import PhRoom_dataset

## 2. Time–Frequency Parameters

These parameters control how the loaded time‑domain RIRs will be processed (FFT size and anti‑aliasing decay). They are passed to the base class constructor.

In [ ]:
samplerate = 48000          # Hz
nfft = samplerate * 3       # FFT size (3 seconds)
alias_decay_db = 0          # No extra anti‑aliasing decay

## 3. Load a Room from DataRES

The `PhRoom_dataset` constructor requires:
- `dataset_directory`: path to the folder where DataRES is stored.
- `room_name`: name of the room to load (as listed in `datasetInfo.json` inside the dataset).

In this example we use the room named `'Otala'`. You should adjust `dataset_directory` to point to your local copy of DataRES.

In [ ]:
dataset_directory = './dataRES'   # Change this to your actual DataRES folder
room_name = 'Otala'

print(f"\nPhRoom_dataset interfaces with DataRES. Loading room '{room_name}' from '{dataset_directory}'...")

physical_room = PhRoom_dataset(
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    dataset_directory=dataset_directory,
    room_name=room_name
)

## 4. Inspect the Loaded Room

The class provides the same attributes as the synthetic rooms: `transducer_number` and `transducer_positions`. Let's examine them.

In [ ]:
print(f"\nThe PhRoom_dataset class is another subclass of the {type(physical_room).__bases__[0].__name__} class.")

print("\nTransducer numbers:")
print(f"  Stage emitters: {physical_room.transducer_number['stg']}")
print(f"  System microphones: {physical_room.transducer_number['mcs']}")
print(f"  System loudspeakers: {physical_room.transducer_number['lds']}")
print(f"  Audience receivers: {physical_room.transducer_number['aud']}")

print("\nTransducer positions (x, y, z in meters):")
print(f"  Stage emitters: \n{physical_room.transducer_positions['stg']}")
print(f"  System microphones: \n{physical_room.transducer_positions['mcs']}")
print(f"  System loudspeakers: \n{physical_room.transducer_positions['lds']}")
print(f"  Audience receivers: \n{physical_room.transducer_positions['aud']}")

## 5. Visualise the Room Setup

The `plot_setup()` method produces a 3D scatter plot of all transducer positions, exactly as in the synthetic case.

In [ ]:
physical_room.plot_setup()
plt.show()

## 6. Energy Coupling and Direct‑to‑Reverberant Ratio

The loaded room provides two additional derived quantities computed from the measured RIRs:
- **Energy coupling**: the total energy (linear sum of squares) of each RIR group, expressed in decibels.
- **Direct‑to‑reverberant ratio (DRR)**: the ratio of direct‑sound energy to reverberant energy, also in decibels.

These are stored in dictionaries `energy_coupling` and `direct_to_reverb_ratio` with keys `'SA'` (stage→audience), `'SM'` (stage→microphones), `'LM'` (loudspeakers→microphones), and `'LA'` (loudspeakers→audience).

In [ ]:
print("\nAll PhRoom subclasses host further useful information about the physical space.")
print("The energy and the direct-to-reverberant ratio (DRR) of the RIRs are contained in the 'energy_coupling' and 'direct_to_reverb_ratio' attributes, respectively.")

print("\nEnergy coupling (dB):")
print(f"  Stage → audience (SA): {mag2db(physical_room.energy_coupling['SA']).item():.2f} dB")
print(f"  Stage → microphones (SM): {mag2db(physical_room.energy_coupling['SM']).item():.2f} dB")
print(f"  Loudspeakers → microphones (LM): {mag2db(physical_room.energy_coupling['LM']).item():.2f} dB")
print(f"  Loudspeakers → audience (LA): {mag2db(physical_room.energy_coupling['LA']).item():.2f} dB")

print("\nDirect-to-reverberant ratio (DRR) (dB):")
print(f"  Stage → audience (SA): {mag2db(physical_room.direct_to_reverb_ratio['SA']).item():.2f} dB")
print(f"  Stage → microphones (SM): {mag2db(physical_room.direct_to_reverb_ratio['SM']).item():.2f} dB")
print(f"  Loudspeakers → microphones (LM): {mag2db(physical_room.direct_to_reverb_ratio['LM']).item():.2f} dB")
print(f"  Loudspeakers → audience (LA): {mag2db(physical_room.direct_to_reverb_ratio['LA']).item():.2f} dB")

## 7. Plot Coupling and DRR

The class includes dedicated methods to visualise these quantities as bar plots.

In [ ]:
physical_room.plot_coupling()
plt.show()

physical_room.plot_DRR()
plt.show()

## 8. Conclusion

You have successfully loaded a measured room from the DataRES dataset and explored its key acoustic properties. The `PhRoom_dataset` class makes it easy to work with real measurement data in the same framework as synthetic rooms.

To try other rooms, check the `datasetInfo.json` file in your DataRES folder and change the `room_name` argument.

For further details, refer to the documentation in `PyRES/physical_room.py` and the DataRES publication.